In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Labs"
)

xpt_file = DATA_DIR / "GLU_D.xpt"
csv_file = DATA_DIR / "GLU_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched original source for fidelity checks
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 GLU_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. STRUCTURE CHECK
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXGLU",
    "LBDGLUSI",
    "LBXIN",
    "LBDINSI",
    "PHAFSTHR",
    "PHAFSTMN"
]


if len(df) != 3352:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 8:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "GLU_D columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: GLU_D structure = "
    "3,352 rows x 8 columns."
)


# ============================================================
# 4. DEFINE LAB / WEIGHT MEASUREMENT FIELDS
# ============================================================

#
# CRITICAL:
#
# These remain exactly as pandas reads them from the XPT.
#
# DO NOT:
# - convert to Int64
# - round()
# - recalculate
# - trim .0
# - use float_format
#

MEASUREMENT_COLUMNS = [
    "WTSAF2YR",
    "LBXGLU",
    "LBDGLUSI",
    "LBXIN",
    "LBDINSI"
]


#
# These are whole-number fasting time fields.
#
CODE_COLUMNS = [
    "PHAFSTHR",
    "PHAFSTMN"
]


# ============================================================
# 5. KNOWN XPORT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)

print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. EXPECTED TINY COUNTS FROM ATTACHED GLU_D
# ============================================================

EXPECTED_TINY_COUNTS = {

    "WTSAF2YR": 482,

    "PHAFSTHR": 45,

    "PHAFSTMN": 58
}


ZERO_ALLOWED_COLUMNS = list(
    EXPECTED_TINY_COUNTS.keys()
)


# ============================================================
# 7. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


allowed_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected = (
    actual_tiny_columns
    -
    allowed_tiny_columns
)


if unexpected:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "GLU_D variables:\n"
        + str(sorted(unexpected))
    )


missing_expected = (
    allowed_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected:

    raise ValueError(
        "STOP: Expected tiny-value variables missing:\n"
        + str(sorted(missing_expected))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


if int(tiny_counts.sum()) != 585:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: All 585 tiny values occur only "
    "in CDC zero-permitted variables."
)


# ============================================================
# 8. CONFIRM LAB RESULTS THEMSELVES HAVE NO TINY ARTIFACT
# ============================================================

LAB_COLUMNS = [
    "LBXGLU",
    "LBDGLUSI",
    "LBXIN",
    "LBDINSI"
]


lab_tiny = int(
    df[LAB_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


if lab_tiny != 0:

    raise ValueError(
        "STOP: Tiny artifact unexpectedly found "
        "inside laboratory result fields."
    )


print(
    "PASS: No tiny artifacts in glucose/insulin "
    "laboratory result fields."
)


# ============================================================
# 9. RESTORE ONLY VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )

    count = int(
        mask.sum()
    )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{count:,} tiny -> 0"
    )


# ============================================================
# 10. VERIFY RESTORED ZERO COUNTS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


zero_failures = []


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: {actual_zero:,}"
    )


    if actual_zero != expected_count:

        zero_failures.append(
            (
                col,
                expected_count,
                actual_zero
            )
        )


if zero_failures:

    raise ValueError(
        "Zero restoration mismatch:\n"
        + str(zero_failures)
    )


print(
    "PASS: All 585 validated zeros restored."
)


# ============================================================
# 11. FINAL TINY CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifacts remain."
    )


print(
    "PASS: No tiny-value artifact remains."
)


# ============================================================
# 12. SEQN VALIDATION
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractions."
    )


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31130
    or
    df["SEQN"].max() != 41474
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 13. FASTING HOURS/MINUTES MUST BE WHOLE NUMBERS
# ============================================================

print("\n--- FASTING TIME INTEGER CHECK ---")


for col in CODE_COLUMNS:

    values = (
        df[col]
        .dropna()
    )


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional.any():

        raise ValueError(
            f"{col} contains unexpected fractional values."
        )


print(
    "PASS: PHAFSTHR and PHAFSTMN "
    "contain whole-number values only."
)


# ============================================================
# 14. CDC WTSAF2YR CHECK
# ============================================================

s = df["WTSAF2YR"]


if not (
    int(s.notna().sum()) == 3352

    and

    int(s.isna().sum()) == 0

    and

    int(s.eq(0).sum()) == 482

    and

    np.isclose(
        s.min(),
        0.0,
        rtol=0,
        atol=0
    )

    and

    np.isclose(
        s.max(),
        327394.027,
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "WTSAF2YR differs from CDC/source."
    )


print(
    "\nPASS: WTSAF2YR validated."
)


# ============================================================
# 15. CDC LAB RANGE / COMPLETENESS CHECKS
# ============================================================

LAB_CHECKS = {

    "LBXGLU": {
        "valid": 3128,
        "missing": 224,
        "min": 45.0,
        "max": 418.0
    },

    "LBDGLUSI": {
        "valid": 3128,
        "missing": 224,
        "min": 2.498,
        "max": 23.203
    },

    "LBXIN": {
        "valid": 3080,
        "missing": 272,
        "min": 1.0,
        "max": 293.58
    },

    "LBDINSI": {
        "valid": 3080,
        "missing": 272,
        "min": 6.0,
        "max": 1761.48
    }
}


print("\n--- CDC LAB CHECKS ---")


for col, spec in LAB_CHECKS.items():

    s = df[col]


    if int(
        s.notna().sum()
    ) != spec["valid"]:

        raise ValueError(
            f"{col}: valid count differs."
        )


    if int(
        s.isna().sum()
    ) != spec["missing"]:

        raise ValueError(
            f"{col}: missing count differs."
        )


    if not np.isclose(
        s.min(),
        spec["min"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col}: minimum differs."
        )


    if not np.isclose(
        s.max(),
        spec["max"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col}: maximum differs."
        )


    print(
        f"PASS: {col:<9}"
        f" range={s.min()}-{s.max()}"
        f" valid={int(s.notna().sum()):,}"
        f" missing={int(s.isna().sum()):,}"
    )


# ============================================================
# 16. CDC FASTING TIME CHECKS
# ============================================================

s = df["PHAFSTHR"]


if not (
    int(s.notna().sum()) == 3251

    and

    int(s.isna().sum()) == 101

    and

    int(s.eq(0).sum()) == 45

    and

    s.min() == 0

    and

    s.max() == 64
):

    raise ValueError(
        "PHAFSTHR differs from CDC/source."
    )


s = df["PHAFSTMN"]


if not (
    int(s.notna().sum()) == 3251

    and

    int(s.isna().sum()) == 101

    and

    int(s.eq(0).sum()) == 58

    and

    s.min() == 0

    and

    s.max() == 59
):

    raise ValueError(
        "PHAFSTMN differs from CDC/source."
    )


print(
    "PASS: PHAFSTHR / PHAFSTMN validated."
)


# ============================================================
# 17. IMPORTANT: DO NOT RECOMPUTE DERIVED LAB VARIABLES
# ============================================================

#
# CDC documents:
#
# LBDGLUSI was derived from LBXGLU.
# LBDINSI was derived from LBXIN.
#
# We intentionally DO NOT recalculate them here.
#
# The released XPT remains the source of truth.
#

print(
    "\nPASS: LBDGLUSI and LBDINSI will be "
    "preserved exactly as released; not recalculated."
)


# ============================================================
# 18. BUILD EXPECTED SOURCE
# ============================================================

#
# Expected source =
#
# original XPT
# +
# ONLY:
#
# WTSAF2YR tiny -> 0
# PHAFSTHR tiny -> 0
# PHAFSTMN tiny -> 0
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 19. CLEAN SEQN
# ============================================================

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


# ============================================================
# 20. FORMAT ONLY FASTING TIME FIELDS
# ============================================================

#
# PHAFSTHR:
# 12.0 -> 12
#
# PHAFSTMN:
# 30.0 -> 30
#
# LAB MEASUREMENTS ARE NOT TOUCHED.
#

for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and fasting time fields "
    "formatted without unnecessary .0."
)


# ============================================================
# 21. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


value_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if (
        col == "SEQN"
        or
        col in CODE_COLUMNS
    ):

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(value_failures)
    )


print(
    "PASS: All 8 variables preserved exactly "
    "after validated zero restoration."
)


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format
# NO round()
# NO recalculation
# NO decimal trimming
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 23. RE-READ WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 24. XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 8 CSV variables numerically "
    "match expected XPT values."
)


# ============================================================
# 25. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\nMissing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 26. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


# ============================================================
# 27. SEQN / FASTING TIME .0 CHECK
# ============================================================

bad_dot_zero = 0
bad_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in [
        "SEQN",
        "PHAFSTHR",
        "PHAFSTMN"
    ]:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            bad_dot_zero += 1


            if len(
                bad_examples
            ) < 20:

                bad_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nSEQN/time values ending '.0':",
    bad_dot_zero
)


if bad_dot_zero != 0:

    print(
        "Examples:",
        bad_examples
    )

    raise ValueError(
        "Unnecessary .0 remains in "
        "SEQN/fasting-time fields."
    )


print(
    "PASS: SEQN and fasting-time fields "
    "have no unnecessary .0."
)


# ============================================================
# 28. STRICT LAB / WEIGHT RAW-TEXT FIDELITY
# ============================================================

#
# Compare actual CSV token to:
#
# str(float(expected XPT value))
#
# Examples:
#
# LBXGLU
# 90.0 -> 90.0
#
# LBDGLUSI
# 4.996 -> 4.996
#
# LBXIN
# 10.0 -> 10.0
#
# WTSAF2YR
# tiny -> 0.0
#
# No beautification / rounding is allowed.
#

measurement_mismatches = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


mismatch_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                float(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            measurement_mismatches[
                col
            ] += 1


            if len(
                mismatch_examples
            ) < 20:

                mismatch_examples.append(
                    (
                        i + 2,
                        col,
                        expected_text,
                        actual_text
                    )
                )


bad_measurements = {
    col: count
    for col, count
    in measurement_mismatches.items()
    if count > 0
}


print(
    "Lab/weight raw-text mismatches:",
    bad_measurements
)


if bad_measurements:

    print(
        "Examples:",
        mismatch_examples
    )

    raise ValueError(
        "Laboratory/weight decimal formatting changed."
    )


print(
    "PASS: WTSAF2YR and all laboratory "
    "measurements appear AS IS in CSV."
)


# ============================================================
# 29. REPORT LEGITIMATE MEASUREMENT .0
# ============================================================

measurement_dot_zero = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


for row in raw_rows:

    for col in MEASUREMENT_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            measurement_dot_zero[
                col
            ] += 1


print(
    "\nLegitimate measurement values ending '.0':"
)


for col, count in measurement_dot_zero.items():

    print(
        f"{col:<10}: {count:,}"
    )


print(
    "\nIMPORTANT: These are lab/weight measurements "
    "and should NOT be changed."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GLU_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Validated tiny -> zero:",
    585
)

print(
    "WTSAF2YR zeros:",
    482
)

print(
    "PHAFSTHR zeros:",
    45
)

print(
    "PHAFSTMN zeros:",
    58
)

print(
    "Lab/weight measurement fields:",
    len(MEASUREMENT_COLUMNS)
)

print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)

print(
    "Missing-position differences:",
    "0 expected"
)

print(
    "Raw lab/weight differences:",
    "0 expected"
)

print(
    "SEQN/time .0:",
    "0 expected"
)

NHANES 2005-2006 GLU_D XPT -> CSV
Rows: 3,352
Columns: 8
PASS: GLU_D structure = 3,352 rows x 8 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 585

Tiny values by variable:
WTSAF2YR    482
PHAFSTMN     58
PHAFSTHR     45

PASS: All 585 tiny values occur only in CDC zero-permitted variables.
PASS: No tiny artifacts in glucose/insulin laboratory result fields.

--- ZERO RESTORATION ---
WTSAF2YR  : 482 tiny -> 0
PHAFSTHR  : 45 tiny -> 0
PHAFSTMN  : 58 tiny -> 0

--- RESTORED ZERO COUNTS ---
WTSAF2YR  : 482
PHAFSTHR  : 45
PHAFSTMN  : 58
PASS: All 585 validated zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny-value artifact remains.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 31130.0
Maximum: 41474.0
PASS: SEQN validated.

--- FASTING TIME INTEGER CHECK ---
PASS: PHAFSTHR and PHAFSTMN contain whole-number values only.

PASS: WTSAF2YR validated.

--- CDC LAB CHECKS ---
PASS: LBXGLU    range=45.0-418.0 valid=3,128 missing=224
PASS: LBD